# Station daily patterns — clustering by altitude and nearest POI

**Goal**: identify whether stations share similar weekday usage patterns, and whether
those patterns correlate with the station's altitude or the category of its nearest POI.

**Method**:
1. Build a 24-value weekday occupancy profile for each station (avg bikes / capacity per hour).
2. Cluster stations with k-means on those profiles.
3. Cross-analyse clusters against altitude and nearest-POI category.

In [ ]:
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

_db_dir = next(
    (p / "src" / "db_upload")
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "db_upload" / "_db.py").exists()
)
sys.path.insert(0, str(_db_dir))
from _db import connect, fetch_all

## 1 · Build hourly occupancy profiles (weekdays only)

In [ ]:
# Aggregates ~10M instants into 556×24 rows — takes ~30s
SQL_PROFILES = """
SELECT
    m.station_id,
    EXTRACT(hour FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid')::int AS hour,
    AVG(getValue(i))::float AS avg_bikes
FROM station_status_mdb m,
LATERAL unnest(instants(bikes_history)) i
WHERE EXTRACT(isodow FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid') BETWEEN 1 AND 5
GROUP BY m.station_id, hour
ORDER BY m.station_id, hour;
"""

conn = connect()
rows = fetch_all(conn, SQL_PROFILES)
conn.close()

profiles_long = pd.DataFrame(rows, columns=["station_id", "hour", "avg_bikes"])
print(f"{profiles_long['station_id'].nunique()} stations, {len(profiles_long)} rows")
profiles_long.head()

In [ ]:
# Pivot to wide format: one row per station, columns = hours 0-23
profiles = (
    profiles_long
    .pivot(index="station_id", columns="hour", values="avg_bikes")
    .dropna()          # drop stations with gaps in any hour
)
profiles.columns.name = None
print(f"Profiles shape: {profiles.shape}  (stations × hours)")

## 2 · Enrich with station metadata and nearest POI

In [ ]:
SQL_META = """
SELECT
    s.station_id,
    s.capacity,
    s.altitude,
    s.elevation_m,
    p.category  AS nearest_poi_category,
    ROUND(
        ST_Distance(
            ST_Transform(s.geom, 25831),
            ST_Transform(p.geom, 25831)
        )::numeric, 1
    ) AS nearest_poi_dist_m
FROM stations s
CROSS JOIN LATERAL (
    SELECT category, geom
    FROM   pois
    ORDER  BY s.geom <-> geom
    LIMIT  1
) p
WHERE s.geom IS NOT NULL;
"""

conn = connect()
meta_rows = fetch_all(conn, SQL_META)
conn.close()

meta = pd.DataFrame(meta_rows, columns=[
    "station_id", "capacity", "altitude", "elevation_m",
    "nearest_poi_category", "nearest_poi_dist_m",
]).set_index("station_id")

# Prefer elevation_m (DEM) when available, fall back to altitude
meta["elev"] = meta["elevation_m"].fillna(meta["altitude"])

print(f"Metadata for {len(meta)} stations")
print(f"Elevation available: {meta['elev'].notna().sum()} stations")
meta.head()

In [ ]:
# Normalise avg_bikes by capacity → occupancy ratio [0, 1]
cap = meta["capacity"].reindex(profiles.index).replace(0, np.nan)
occ = profiles.div(cap, axis=0).dropna()

# Keep only stations that appear in both tables
common = occ.index.intersection(meta.index)
occ  = occ.loc[common]
meta = meta.loc[common]

print(f"Working set: {len(occ)} stations")

## 3 · Choose number of clusters (elbow)

In [ ]:
X = StandardScaler().fit_transform(occ)

inertias = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias[k] = km.inertia_

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(list(inertias.keys()), list(inertias.values()), marker="o")
ax.set_xlabel("k")
ax.set_ylabel("Inertia")
ax.set_title("Elbow — choose k")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4 · Fit k-means and label stations

In [ ]:
K = 4  # adjust after inspecting the elbow

km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = km.fit_predict(X)

meta["cluster"] = labels
occ["cluster"]  = labels

print("Stations per cluster:")
print(meta["cluster"].value_counts().sort_index().to_string())

## 5 · Mean occupancy profile per cluster

In [ ]:
hours = list(range(24))
palette = plt.get_cmap("tab10")

fig, axes = plt.subplots(1, K, figsize=(4 * K, 4), sharey=True)
for c in range(K):
    ax = axes[c]
    grp = occ[occ["cluster"] == c].drop(columns="cluster")
    mean = grp.mean()
    std  = grp.std()
    ax.fill_between(hours, mean - std, mean + std, alpha=0.2, color=palette(c))
    ax.plot(hours, mean, color=palette(c), linewidth=2)
    ax.set_title(f"Cluster {c}  (n={len(grp)})", fontsize=11)
    ax.set_xlabel("Hour of day")
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Occupancy (bikes / capacity)")
fig.suptitle("Mean weekday occupancy profile per cluster", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6 · Altitude distribution per cluster

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

data_by_cluster = [
    meta.loc[meta["cluster"] == c, "elev"].dropna().values
    for c in range(K)
]
bp = ax.boxplot(data_by_cluster, patch_artist=True, medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bp["boxes"], range(K)):
    patch.set_facecolor(palette(c))
    patch.set_alpha(0.7)

ax.set_xticklabels([f"Cluster {c}" for c in range(K)])
ax.set_ylabel("Elevation (m)")
ax.set_title("Elevation by cluster")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7 · Nearest POI category distribution per cluster

In [ ]:
ct = (
    meta.groupby(["cluster", "nearest_poi_category"])
    .size()
    .unstack(fill_value=0)
)
# Normalise to proportions within each cluster
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 4))
ct_pct.T.plot(kind="bar", ax=ax, colormap="tab10", width=0.7)
ax.set_xlabel("Nearest POI category")
ax.set_ylabel("% of stations in cluster")
ax.set_title("Nearest POI category distribution by cluster")
ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8 · Map — stations coloured by cluster

In [ ]:
import folium

conn = connect()
coords = fetch_all(conn, "SELECT station_id, lat, lon FROM stations WHERE geom IS NOT NULL")
conn.close()
coords_df = pd.DataFrame(coords, columns=["station_id", "lat", "lon"]).set_index("station_id")

CLUSTER_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#ff7f00",
                  "#984ea3", "#a65628", "#f781bf", "#999999"]

m = folium.Map(location=[41.39, 2.15], zoom_start=13)

joined = coords_df.join(meta[["cluster"]], how="inner")
for sid, row in joined.iterrows():
    c = int(row["cluster"])
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=CLUSTER_COLORS[c],
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Station {sid} · Cluster {c}",
    ).add_to(m)

# Legend
legend_html = "<div style='position:fixed;bottom:30px;left:30px;background:white;padding:10px;border-radius:6px;font-size:13px'>"
for c in range(K):
    legend_html += f"<span style='background:{CLUSTER_COLORS[c]};display:inline-block;width:12px;height:12px;border-radius:50%;margin-right:4px'></span>Cluster {c}<br>"
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

m